###Eliminamos los archivos que se cargaron en el dia de la ejecucion y se cargan en la carpeta backup

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
--- crear widgets
create widget text storageName default "adlsvinkosetl";


In [0]:
%python
storageName = dbutils.widgets.get("storageName")


In [0]:
archivos = dbutils.fs.ls(
    f"abfss://raw@{storageName}.dfs.core.windows.net/"
)

display(archivos)

[]

In [0]:
#creamos ZIP

from datetime import datetime
nombre_zip = datetime.now().strftime("%Y%m%d_%H%M%S") + ".zip"
print(nombre_zip)

20260708_150533.zip


In [0]:
#Descargamos txt

import os
ruta_local="/tmp/txt"
os.makedirs(ruta_local,exist_ok=True)

for f in archivos:
    if f.name.endswith(".txt"):
        dbutils.fs.cp(
            f.path,
            "file:" + ruta_local + "/" + f.name
        )

In [0]:
#Creamos el .zio con los txt

import zipfile

ruta_zip="/tmp/" + nombre_zip
with zipfile.ZipFile(
        ruta_zip,
        "w",
        zipfile.ZIP_DEFLATED
    ) as zipf:

    for archivo in os.listdir(ruta_local):
        zipf.write(
            os.path.join(ruta_local,archivo),
            archivo
        )

In [0]:
#enviamos el zip al bkp

dbutils.fs.cp(
    "file:" + ruta_zip,
    f"abfss://backup@{storageName}.dfs.core.windows.net/{nombre_zip}"
)

#Borramos los archivos de raw

for f in archivos:
    if f.name.endswith(".txt"):
        dbutils.fs.rm(f.path)